<a href="https://colab.research.google.com/github/dataprogpy/code-samples/blob/main/starter_files/09_time_series_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Time Series Analysis

In [2]:
import datetime as dt

import polars as pl
import pandas as pd
import yfinance as yf

import altair as alt


## Data Wrangling and Exploratory Data Analysis

In [2]:
df_missing = pl.DataFrame({
    "date": [dt.date(2024, 1, 1), dt.date(2024, 1, 2), dt.date(2024, 1, 3)],
    "value": [10, None, 15]
})

df = (
    df_missing
    .with_columns(
        pl.col("value")
        .fill_null(strategy="forward")
        .alias("value_forward")
        )
    .with_columns(
        pl.col("value")
        .fill_null(strategy="backward")
        .alias("value_backward")
    )
    .with_columns(
        pl.col("value")
        .fill_null(strategy="mean")
        .alias("value_mean")
    )
)
df

date,value,value_forward,value_backward,value_mean
date,i64,i64,i64,i64
2024-01-01,10,10,10,10
2024-01-02,null,10,15,12
2024-01-03,15,15,15,15


In [3]:
# Example DataFrame with daily data
df_daily = pl.DataFrame({
    "date": pl.date_range(dt.date(2024, 1, 1), dt.date(2024, 2, 15), "1d", eager=True),
    "sales": range(46)
})

# Downsample to monthly average sales
df_monthly = df_daily.group_by_dynamic(
    "date", every="1mo"
).agg(
    pl.col("sales").mean().alias("average_sales")
)
display(df_monthly)

date,average_sales
date,f64
2024-01-01,15.0
2024-02-01,38.0


In [16]:
# Define the ticker symbol and date range
ticker = 'GOOG'
start_date = '2019-01-01'
end_date = '2025-06-01'

# Download the data using yfinance
df = yf.download(ticker, start=start_date, end=end_date)

# Display the first few rows of the DataFrame
display(df.head())
df_pl = pl.from_pandas(df, include_index=True )
df_pl.columns

/tmp/ipython-input-16-3659620356.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,GOOG,GOOG,GOOG,GOOG,GOOG
Date,,,,,
2019-01-02,51.983505,52.305095,50.485410,50.528155,30652000
2019-01-03,50.502804,52.536712,50.403889,51.742433,36822000
2019-01-04,53.219154,53.225616,51.067345,51.324419,41878000
2019-01-07,53.103844,53.382689,52.426371,53.258427,39638000
2019-01-08,53.496010,53.907565,52.713165,53.487561,35298000


['Date',
 "('Close', 'GOOG')",
 "('High', 'GOOG')",
 "('Low', 'GOOG')",
 "('Open', 'GOOG')",
 "('Volume', 'GOOG')"]

In [17]:
def colname(col: str):
  if col.startswith('('):
        return ( col
          .lstrip('(')
          .rstrip(')')
          .split(",")[0]
          .strip("'")
          .lower()
        )
  return col.lower()

print([colname(col) for col in df_pl.columns])

df_pl = df_pl.rename(colname)
df_pl.head()

['date', 'close', 'high', 'low', 'open', 'volume']


date,close,high,low,open,volume
datetime[ns],f64,f64,f64,f64,i64
2019-01-02 00:00:00,51.983505,52.305095,50.48541,50.528155,30652000
2019-01-03 00:00:00,50.502804,52.536712,50.403889,51.742433,36822000
2019-01-04 00:00:00,53.219154,53.225616,51.067345,51.324419,41878000
2019-01-07 00:00:00,53.103844,53.382689,52.426371,53.258427,39638000
2019-01-08 00:00:00,53.49601,53.907565,52.713165,53.487561,35298000


In [5]:
alt.Chart(df_pl).mark_line().encode(
    x='date',
    y='close',
    tooltip=['date', 'close']
).properties(
    width=800,
    height=400
)

alt.Chart(...)

### Filtering data based on temporal properties

In [6]:
alt.Chart(
    df_pl.filter(
         pl.col("date").is_between(
             dt.datetime(2020, 1, 1),
             dt.datetime(2022, 12, 31)
             )
    )
    ).mark_line().encode(
    x='date',
    y='close',
    tooltip=['date', 'close']
).properties(
    width=800,
    height=400
)

alt.Chart(...)

In [18]:
# Calculate 30-day and 90-day moving averages
df_with_ma = df_pl.with_columns(
    pl.col("close").rolling_mean(window_size=30).alias("ma_30_day"),
    pl.col("close").rolling_mean(window_size=90).alias("ma_90_day")
)
# Plot the daily closing price
base = alt.Chart(df_with_ma).encode(x='date:T')

closing_price = base.mark_line().encode(
    y=alt.Y('close:Q', title='Closing Price (USD)')
).properties(
    title='Google (GOOG) Daily Stock Price',
    width=800,
    height=400,
)

# Create layers for the moving averages
ma_30 = base.mark_line(color='orange').encode(
    y='ma_30_day:Q'
)

ma_90 = base.mark_line(color='red').encode(
    y='ma_90_day:Q'
)

# Combine the original price plot with the moving average plots
(closing_price + ma_30 + ma_90)

alt.LayerChart(...)

## Visual Exploration of Trend and Seasonlity

In [96]:
df_ap = pl.read_csv("/content/drive/MyDrive/dataprogpy/data/AirPassengers.csv")

df_ap = df_ap.with_columns(
    pl.col("Month").str.strptime(pl.Date, format="%Y-%m").alias("month_dt"),
    pl.col("#Passengers").rolling_mean(window_size=12).alias("ma_12_month"),
).with_columns(
    pl.col("month_dt").dt.month().alias("season"),
    (pl.col("#Passengers")/pl.col("ma_12_month")).alias("detrended")
).select(
    pl.col("month_dt","season","ma_12_month","detrended" ),
    pl.col("#Passengers").alias("passengers")
)

df_ap.tail()

month_dt,season,ma_12_month,detrended,passengers
date,i8,f64,f64,i64
1960-08-01,8,463.333333,1.307914,606
1960-09-01,9,467.083333,1.0876,508
1960-10-01,10,471.583333,0.977558,461
1960-11-01,11,473.916667,0.822929,390
1960-12-01,12,476.166667,0.907245,432


In [98]:
base = alt.Chart(df_ap)

full = base.mark_line().encode(
    alt.X('month_dt:T'),
    alt.Y('passengers:Q'),
    tooltip=['month_dt', 'passengers']
).properties(
    width=800,
    height=400
)

ma_12_month = full.mark_line(color='orange').encode(
    alt.Y('ma_12_month:Q'),
)

detrended = full.mark_line(color='red').encode(
    alt.Y('detrended:Q').scale(zero=False)
)

seasonlity = (
    full
    .mark_boxplot()
    .encode(
        alt.X('season:O'),
        alt.Y('detrended:Q').scale(zero=False),
    )
)

# (base + ma_12_month + detrended)
display(full + ma_12_month )
display(detrended)
display(seasonlity)

alt.LayerChart(...)

alt.Chart(...)

alt.Chart(...)